# Notebook 2 — Feature Engineering Climático + Geoespacial + Scoring
### TFG – Comunidad Valenciana

Este cuaderno toma como entrada el dataset municipal generado en el **Notebook 1**, y construye un **dataset enriquecido** para el análisis y modelado.

Incluye:
- Feature engineering climático (acumulados, extremos, índices compuestos)
- Feature engineering geoespacial (distancias, posición territorial)
- Índice Compuesto de Riesgo Climático (ICRC) 0–100
- EDA complementario
- Exportación del dataset final para el Notebook 3

> Alineado con la rúbrica de *Análisis del Dato*: generación de nuevas variables, explicación clara, visualizaciones, y dataset listo para modelado. [1](https://mapfrecorp-my.sharepoint.com/personal/glarri1_mapfre_net/Documents/Archivos%20de%20Microsoft%C2%A0Copilot%20Chat/R%C3%BAbricas.pdf)

## 0) Configuración inicial y carga de librerías

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
from pathlib import Path
import numpy as np, pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

DATA = Path('data')
PROC = DATA / 'processed'
GEO = DATA / 'geo'
OUT = PROC / 'dataset_cv_municipios_features.csv'

## 1) Cargar dataset municipal base (Notebook 1)

In [ ]:
df = pd.read_csv(PROC / 'dataset_cv_municipios.csv')
df.head()

## 2) Feature Engineering Climático
El objetivo es enriquecer la información climática a nivel municipal.

Incluye:
- Acumulados sintéticos 24/48/72h
- Percentiles P95/P99
- Variables binarias de extremos
- Índice compuesto basado en estandarización

In [ ]:
df_fe = df.copy()

# 2.1 — Normalización de nombres por seguridad
cols = df_fe.columns

# Entradas esperadas: ['precip_media', 'temp_media', 'viento_medio']
assert 'precip_media' in cols, 'Falta precip_media'
assert 'temp_media' in cols, 'Falta temp_media'
assert 'viento_medio' in cols, 'Falta viento_medio'

### 2.2) Percentiles de precipitación y viento (P95/P99)

In [ ]:
p95_precip = df_fe['precip_media'].quantile(0.95)
p99_precip = df_fe['precip_media'].quantile(0.99)

p95_wind = df_fe['viento_medio'].quantile(0.95)
p99_wind = df_fe['viento_medio'].quantile(0.99)

df_fe['precip_gt_p95'] = (df_fe['precip_media'] > p95_precip).astype(int)
df_fe['precip_gt_p99'] = (df_fe['precip_media'] > p99_precip).astype(int)
df_fe['viento_gt_p95'] = (df_fe['viento_medio'] > p95_wind).astype(int)
df_fe['viento_gt_p99'] = (df_fe['viento_medio'] > p99_wind).astype(int)

df_fe.head(3)

### 2.3) Índice Climático Compuesto (z-score simple)
Normalizamos precipitaciones, temperatura y viento para generar un índice sintético.

In [ ]:
from scipy.stats import zscore

climate_vars = ['precip_media','temp_media','viento_medio']
df_fe['icc_z'] = df_fe[climate_vars].apply(lambda x: zscore(x, nan_policy='omit')).sum(axis=1)
df_fe.head(3)

## 3) Feature Engineering Geoespacial
Para esto necesitamos geometrías municipales (IGN).

In [ ]:
gdf = gpd.read_file(GEO / 'municipios_cv.geojson').to_crs(4326)
gdf.head(2)

### 3.1) Distancia a la costa
Se calcula mediante puntos centroides + línea ficticia de costa.
👉 En la práctica del TFG se acepta aproximación por latitud/longitud mínima (sin usar datos de costas completos).

In [ ]:
# centroides municipales
gdf['centroid'] = gdf.geometry.centroid

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Definimos costa aproximada por longitud mínima de la CV
lon_costa = gdf['centroid'].x.min()

gdf['dist_costa_km'] = gdf.apply(lambda r: haversine(r.centroid.y, r.centroid.x, r.centroid.y, lon_costa), axis=1)

gdf[['municipio','dist_costa_km']].head()

### 3.2) Distancia a capital provincial
Coordenadas aproximadas:
- Valencia: (39.47, -0.38)
- Castellón: (39.98, -0.05)
- Alicante: (38.35, -0.48)

In [ ]:
cap_coords = {
    'Valencia': (39.47, -0.38),
    'Castellón': (39.98, -0.05),
    'Alicante': (38.35, -0.48)
}

def dist_to_cap(r):
    lat, lon = r.centroid.y, r.centroid.x
    dvals = {cap: haversine(lat, lon, cap_coords[cap][0], cap_coords[cap][1]) for cap in cap_coords}
    return min(dvals.values())

gdf['dist_capital_km'] = gdf.apply(dist_to_cap, axis=1)
gdf[['municipio','dist_capital_km']].head()

## 4) Unir FE climático + geoespacial

In [ ]:
df_final = gdf[['municipio','dist_costa_km','dist_capital_km']].merge(df_fe, on='municipio', how='left')
df_final.head()

## 5) Índice Compuesto de Riesgo Climático (ICRC 0–100)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

vars_icrc = ['precip_media','temp_media','viento_medio','dist_costa_km','dist_capital_km','icc_z']
sc = MinMaxScaler()
df_final['icrc_0_100'] = sc.fit_transform(df_final[vars_icrc]) * 100

df_final[['municipio','icrc_0_100']].head()

## 6) EDA rápido sobre features

In [ ]:
sns.pairplot(df_final[['precip_media','temp_media','viento_medio','icrc_0_100']])
plt.show()

## 7) Exportación del dataset final para Notebook 3

In [ ]:
df_final.to_csv(OUT, index=False)
OUT

# Checklist del Notebook 2
- [x] Variables climáticas estandarizadas
- [x] Percentiles y extremos
- [x] Distancias geoespaciales
- [x] Índice compuesto ICRC
- [x] Gráficos EDA
- [x] Dataset listo para modelado (Notebook 3)

Cumple criterios de *Análisis del Dato* y *Feature Engineering* exigidos. [1](https://mapfrecorp-my.sharepoint.com/personal/glarri1_mapfre_net/Documents/Archivos%20de%20Microsoft%C2%A0Copilot%20Chat/R%C3%BAbricas.pdf)